# Phase 2 Implications
### Using this file to start off cleanly for second phase of House Pricing Prediction problem

1. Used Pipeline To Perform Preprocessing in Folds
2. Apply XGBoost Model With Proper Parameters
3. Keep this file clean and import neccessary data from the main file

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [2]:
train_data = pd.read_csv("data/train.csv")
test_data = pd.read_csv("data/test.csv")

### Simplifying Logic Of Filling NaNs of LotFrontage And will use as an Imputer in Folds

In [3]:
def compute_lot_frontage_map(X):
    """
    Learn LotFrontage imputation statistics from X.
    Returns:
        group_map    : dict {(Neighborhood, LotShape) -> median}
        neigh_map    : dict {Neighborhood -> median}  (fallback)
        global_median: float  (final fallback)
    """
    known = X[X["LotFrontage"].notna()]

    # Primary: group by Neighborhood + LotShape
    group_map = (
        known
        .groupby(["Neighborhood", "LotShape"])["LotFrontage"]
        .median()
        .to_dict()
    )

    # Fallback 1: Neighborhood only
    neigh_map = (
        known
        .groupby("Neighborhood")["LotFrontage"]
        .median()
        .to_dict()
    )

    # Fallback 2: global median
    global_median = known["LotFrontage"].median()

    return group_map, neigh_map, global_median

In [4]:
def fill_lot_frontage(X, group_map, neigh_map, global_median):
    """
    Fill LotFrontage NaNs using learned maps.
    Priority: group median → neighborhood median → global median
    """
    X = X.copy()

    def impute_row(row):
        if pd.notna(row["LotFrontage"]):
            return row["LotFrontage"]

        group_key = (row["Neighborhood"], row["LotShape"])
        if group_key in group_map:
            return group_map[group_key]

        if row["Neighborhood"] in neigh_map:
            return neigh_map[row["Neighborhood"]]

        return global_median

    X["LotFrontage"] = X.apply(impute_row, axis=1)
    return X

In [5]:
from sklearn.base import BaseEstimator, TransformerMixin

class LotFrontageImputer(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        self.group_map_, self.neigh_map_, self.global_median_ = \
            compute_lot_frontage_map(X)
        return self

    def transform(self, X):
        return fill_lot_frontage(
            X,
            self.group_map_,
            self.neigh_map_,
            self.global_median_
        )

### Filling data on full dataset in which we are not applying statistics and have fixed rules

In [6]:
def apply_fixed_rules(df):
    df = df.copy()
    # Basement
    mask_no_bsmt = df["TotalBsmtSF"] == 0
    for col in ["BsmtQual","BsmtCond","BsmtFinType1","BsmtExposure","BsmtFinType2"]:
        df.loc[mask_no_bsmt, col] = "NA"
    # Garage
    mask_no_garage = df["GarageArea"] == 0
    for col in ["GarageType","GarageFinish","GarageQual","GarageCond"]:
        df.loc[mask_no_garage, col] = "NA"
    df.loc[mask_no_garage, "GarageYrBlt"] = 0
    # MasVnr
    df.loc[df["MasVnrArea"] == 1.0, "MasVnrArea"] = 0.0
    df.loc[df["MasVnrArea"] == 0, "MasVnrType"] = "NA"
    df.loc[df["MasVnrArea"].isna(), "MasVnrArea"] = 0
    df.loc[df["MasVnrType"].isna(), "MasVnrType"] = "BrkFace"
    # Others
    df.loc[df["PoolArea"] == 0, "PoolQC"] = "NA"
    df.loc[df["MiscVal"] == 0, "MiscFeature"] = "NA"
    df.loc[df["Fireplaces"] == 0, "FireplaceQu"] = "NA"
    df.loc[df["Alley"].isna(), "Alley"] = "NA"
    df.loc[df["Fence"].isna(), "Fence"] = "NA"
    df.loc[df["Electrical"].isna(), "Electrical"] = "SBrkr"
    return df

### Converting categorical data into Ordinal and Binary since they will be fixed

In [7]:
def apply_encodings(df):
    df = df.copy()

    # ── Binary ────────────────────────────────────────────
    df['Street']     = df['Street'].map({'Grvl': 0, 'Pave': 1})
    df['CentralAir'] = df['CentralAir'].map({'N': 0, 'Y': 1})

    # ── Ordinal ───────────────────────────────────────────

    # Quality scale (shared across multiple columns)
    quality_map = {'NA': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
    quality_cols = [
        'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
        'HeatingQC', 'KitchenQual', 'FireplaceQu',
        'GarageQual', 'GarageCond', 'PoolQC'
    ]
    for col in quality_cols:
        df[col] = df[col].fillna('NA').map(quality_map)

    df['LotShape'] = df['LotShape'].map(
        {'IR3': 1, 'IR2': 2, 'IR1': 3, 'Reg': 4}
    )
    df['Utilities'] = df['Utilities'].map(
        {'ELO': 1, 'NoSeWa': 2, 'NoSewr': 3, 'AllPub': 4}
    )
    df['LandSlope'] = df['LandSlope'].map(
        {'Sev': 1, 'Mod': 2, 'Gtl': 3}
    )
    df['BsmtExposure'] = df['BsmtExposure'].fillna('NA').map(
        {'NA': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4}
    )
    df['BsmtFinType1'] = df['BsmtFinType1'].fillna('NA').map(
        {'NA': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6}
    )
    df['BsmtFinType2'] = df['BsmtFinType2'].fillna('NA').map(
        {'NA': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6}
    )
    df['GarageFinish'] = df['GarageFinish'].fillna('NA').map(
        {'NA': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3}
    )
    df['PavedDrive'] = df['PavedDrive'].map(
        {'N': 1, 'P': 2, 'Y': 3}
    )
    df['Functional'] = df['Functional'].map(
        {'Sal': 1, 'Sev': 2, 'Maj2': 3, 'Maj1': 4,
         'Mod': 5, 'Min2': 6, 'Min1': 7, 'Typ': 8}
    )

    return df

In [8]:
X = train_data.drop(columns=["SalePrice"])
y = np.log1p(train_data["SalePrice"])

X         = apply_fixed_rules(X)
test_data = apply_fixed_rules(test_data)

X         = apply_encodings(X)
test_data = apply_encodings(test_data)

### Converting nominal columns with OneHotEncoder inside Pipeline

In [9]:
nominal_cols = [
    'MSZoning', 'Alley', 'LandContour', 'LotConfig',
    'Neighborhood', 'Condition1', 'Condition2',
    'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl',
    'Exterior1st', 'Exterior2nd', 'MasVnrType',
    'Foundation', 'Heating', 'Electrical',
    'GarageType', 'Fence', 'MiscFeature',
    'SaleType', 'SaleCondition'
]

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("onehot", OneHotEncoder(handle_unknown="ignore"), nominal_cols)
    ],
    remainder="passthrough"  # ← all other columns pass through untouched
)

### Applying Model With A Pipeline to have Imputer and Encoder Inside

In [11]:
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

pipeline = Pipeline([
    ("lot_frontage", LotFrontageImputer()),
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42
    ))
])

In [13]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor

# Create stratification bins
y_bins = pd.qcut(y, q=10, labels=False)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_rmse_log = []
val_rmse_log = []

train_rmse_actual = []
val_rmse_actual = []

all_actual = []
all_predicted = []
rows = []

# -----------------------------
# Cross Validation Loop
# -----------------------------
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_bins), 1):

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    pipeline.fit(
        X_train,
        y_train,
    )

    # Predictions (log scale)
    y_train_pred_log = pipeline.predict(X_train)
    y_val_pred_log = pipeline.predict(X_val)

    # Convert back to actual scale
    y_train_actual = np.expm1(y_train)
    y_val_actual = np.expm1(y_val)

    y_train_pred = np.expm1(y_train_pred_log)
    y_val_pred = np.expm1(y_val_pred_log)

    # -------- LOG SCALE RMSE --------
    rmse_train_log = np.sqrt(mean_squared_error(y_train, y_train_pred_log))
    rmse_val_log = np.sqrt(mean_squared_error(y_val, y_val_pred_log))

    train_rmse_log.append(rmse_train_log)
    val_rmse_log.append(rmse_val_log)

    # -------- ACTUAL SCALE RMSE --------
    rmse_train_actual = np.sqrt(mean_squared_error(y_train_actual, y_train_pred))
    rmse_val_actual = np.sqrt(mean_squared_error(y_val_actual, y_val_pred))

    train_rmse_actual.append(rmse_train_actual)
    val_rmse_actual.append(rmse_val_actual)

    # Store predictions
    for idx, actual, pred in zip(
        X_val.index, y_val_actual.values, y_val_pred
    ):
        rows.append([idx, actual, round(pred, 2), round(actual - pred, 2), fold])

    all_actual.extend(y_val_actual.values)
    all_predicted.extend(y_val_pred)

    print(f"Fold {fold}")
    print(f"Train RMSE (Log): {rmse_train_log:.4f}")
    print(f"Val   RMSE (Log): {rmse_val_log:.4f}")
    print(f"Train RMSE (Actual): {rmse_train_actual:.2f}")
    print(f"Val   RMSE (Actual): {rmse_val_actual:.2f}")
    print("-" * 40)

# -----------------------------
# Summary
# -----------------------------
print("Average Train RMSE (Log):", np.mean(train_rmse_log))
print("Average Val   RMSE (Log):", np.mean(val_rmse_log))
print("Average Train RMSE (Actual):", np.mean(train_rmse_actual))
print("Average Val   RMSE (Actual):", np.mean(val_rmse_actual))

Fold 1
Train RMSE (Log): 0.0365
Val   RMSE (Log): 0.1140
Train RMSE (Actual): 6790.92
Val   RMSE (Actual): 20894.25
----------------------------------------
Fold 2
Train RMSE (Log): 0.0335
Val   RMSE (Log): 0.1422
Train RMSE (Actual): 6359.59
Val   RMSE (Actual): 25301.46
----------------------------------------
Fold 3
Train RMSE (Log): 0.0365
Val   RMSE (Log): 0.1126
Train RMSE (Actual): 6737.36
Val   RMSE (Actual): 27430.44
----------------------------------------
Fold 4
Train RMSE (Log): 0.0371
Val   RMSE (Log): 0.1045
Train RMSE (Actual): 6988.29
Val   RMSE (Actual): 29274.85
----------------------------------------
Fold 5
Train RMSE (Log): 0.0361
Val   RMSE (Log): 0.1594
Train RMSE (Actual): 6799.32
Val   RMSE (Actual): 44110.13
----------------------------------------
Average Train RMSE (Log): 0.035936585575311056
Average Val   RMSE (Log): 0.12654464871230986
Average Train RMSE (Actual): 6735.096633613661
Average Val   RMSE (Actual): 29402.224273743515
